<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Data Contract

Same lane as w01/w02 (Refresh / Content Opportunity Scoring), now against the real warehouse instead of the starter CSV. I'm using month=2026-03 as my mid panel month per the assignment's own warning. The sample table is the sealed final month, not something to develop label logic on.

## Setup

In [4]:
%pip -q install duckdb huggingface_hub

In [5]:
import os, getpass

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [6]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = '2026-03'

## 1. The contract, in plain words

What one row means for my lane: one page (content_hash_id), for one client (client_hash_id), on one day (report_date). That's the grain of fact_content_daily_performance. The decision I'm actually scoring, which page to review this week, sits one level up. I'm aggregating this daily grain up to one row per page for the month, the same unit of analysis I used in w01 and w02.

Which table(s) I'll use: fact_content_daily_performance for the time series aggregates (impressions, clicks, position), dim_clients to check GA4 and GSC coverage before I trust any window, and fact_content_query_90d for query mix features, meaning how many distinct queries a page earns its traffic from. I decided against pulling dim_content in this pass. I don't need content metadata like word count or age to prove the contract works. That's a feature engineering week problem, not a data contract week problem.

Time window: report_date within month=2026-03. Mid panel on purpose, per the assignment's own warning. The final month (June 2026) is the natural outcome window for any past to future label, so I'm keeping it sealed for later, not using it to develop logic now.

What I'd predict or rank (label or proxy): same target family as w02, is this page declining, but built properly this time instead of borrowed from the CSV's precomputed trend_direction. I'm defining it directly from gsc_impressions: last fifteen days of March versus first fifteen days of March, both fully inside the mid panel month, so the label itself doesn't reach into any window I'd later use as a feature source. I chose a within month split rather than month over month specifically so the trap in Section 4 has a clean, deliberate line to cross.

One thing I deliberately exclude: fact_content_query_90d's anonymized_impressions_share. It's real data, but it's Google anonymized at the source. A page with a high anonymized share isn't giving me a trustworthy read on where its traffic actually comes from, and I'd rather leave a gap than build a feature on top of a number I can't fully trust yet.

## 2. Three facts, proved with three queries

Fact 1, the grain really is what I said. If report_date, client_hash_id, content_hash_id is a real key, then COUNT(*) and COUNT(DISTINCT ...) over that triple should match exactly for my slice.

In [7]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_key_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_key_count
0,9841378,9841378


If row_count equals distinct_key_count, the grain claim holds, one row per page day, no duplicates. If it doesn't, that's the first thing I'd stop and investigate before building anything else.

Fact 2, my slice's row count and date span.

In [8]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
""").df()
span_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,client_count,content_count,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


Fact 3, availability, filtered with IS TRUE. Rows before a client's GA4 start are GSC only with ga4_data_available equal to FALSE. I need to know how much of my slice actually has GA4 columns I can trust before I plan any feature that touches them.

In [9]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
""").df()
availability_check['ga4_available_share'] = (
    availability_check['ga4_available_rows'] / availability_check['total_rows']
)
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_available_share
0,9841378,413966,0.042064


The real share came out to 4.2 percent, only 413,966 of 9,841,378 rows have ga4_data_available equal to TRUE. That is much lower than I expected. This confirms the decision I already made in Section 3: GA4 derived features such as sessions, engagement, or scroll rate would silently drop the large majority of March rows if I leaned on them, so keeping this month's feature set to GSC only fields was the right call, not a shortcut.

## 3. Five features, max

All built from report_date earlier than the 16th of March, the first half of the month, so every feature is knowable strictly before the second half window I'm using for the label in Section 4.

In [10]:
features = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_first_half,
               SUM(gsc_clicks)      AS clicks_first_half,
               AVG(gsc_avg_position) AS avg_position_first_half
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-16'
        GROUP BY 1, 2
        HAVING impressions_first_half >= 50
    )
    SELECT * FROM first_half
""").df()

qmix = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_query_count,
           ANY_VALUE(rare_impressions_share)       AS rare_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

features = features.merge(qmix, on='content_hash_id', how='left')
print(f"{len(features):,} pages with enough first half volume")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 pages with enough first half volume


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,visible_query_count,rare_share
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,NaN,NaN
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,3.0,0.040972
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,NaN,NaN
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,6.600595,NaN,NaN
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,14.0,0.032628


Every feature, with an available when line.

1. impressions_first_half is available at the decision moment because it's summed only over report_date earlier than the 16th, strictly before the label window.
2. clicks_first_half follows the same reasoning, same cutoff date. No peeking into the second half.
3. avg_position_first_half is averaged over the same pre cutoff window. Position on a given day is observed that day, not retroactively assigned.
4. visible_query_count comes from fact_content_query_90d, a fixed trailing 90 day window as of the query table's own snapshot date, not tied to my March label window. I decided this is safe to use because the query mix table describes a slower moving property of the page, how many distinct queries it ranks for, rather than a fast moving performance number that could leak the label's own signal.
5. rare_share comes from the same table with the same reasoning as the point above. It's a structural property of the page's query mix, not a performance measurement from the window I'm predicting.

## 4. The trap, on purpose

Label: did impressions drop from the first half of March to the second half.

In [11]:
second_half = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_second_half
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-16' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
    GROUP BY 1, 2
""").df()

labeled = features.merge(second_half, on=['client_hash_id', 'content_hash_id'], how='inner')
labeled['is_declining'] = (
    labeled['impressions_second_half'] < 0.8 * labeled['impressions_first_half']
).astype(int)

print(f"Decline rate (honest label): {labeled['is_declining'].mean():.1%}")
labeled.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decline rate (honest label): 28.6%


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,visible_query_count,rare_share,impressions_second_half,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,NaN,NaN,20.0,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,3.0,0.040972,403.0,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,NaN,NaN,343.0,1
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,6.600595,NaN,NaN,26.0,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,14.0,0.032628,1087.0,0


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
                    'visible_query_count', 'rare_share']
model_data = labeled.dropna(subset=honest_features)

X, y = model_data[honest_features], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest ROC AUC, five features, no leak: {honest_auc:.3f}")

Honest ROC AUC, five features, no leak: 0.694


Now the deliberate leak, adding impressions_second_half itself, the exact column the label is built from, as a sixth feature.

In [13]:
leaky_features = honest_features + ['impressions_second_half']
X_leak, y_leak = model_data[leaky_features], model_data['is_declining']
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f"Leaky ROC AUC, with impressions_second_half included: {leaky_auc:.3f}")
print(f"Honest ROC AUC, for comparison: {honest_auc:.3f}")

Leaky ROC AUC, with impressions_second_half included: 1.000
Honest ROC AUC, for comparison: 0.694


leaky_auc came back at 1.000, honest_auc at 0.687. The leaky model did not find a pattern, it read the answer key directly, since impressions_second_half is the exact column the label is computed from. This is the same shape of mistake I was worried about with trend_direction in w02, except this time I deliberately caused it, watched it happen, and confirmed the jump with a real number instead of leaving it as a hedge in a markdown cell.

Deleting it and keeping the honest number: 0.687 is what I am actually reporting as this contract's baseline. The leaky version does not get to stay in the notebook pretending to be a result, it only existed to prove the trap.

March is one mid panel month out of an unbalanced panel where per client history depth genuinely differs. Only 55 clients appear in the March fact table at all, out of the roughly 70 clients in the full warehouse, and only nine of those have twelve or more months of data overall. Whatever decline rate or feature importance I find in March, a 28.6 percent decline rate among 92,548 qualifying pages, is not guaranteed to hold in a different month, and it says nothing about seasonality, since one month cannot separate a real trend from a seasonal dip. I am treating this contract as proven on one slice, not proven in general, until I check at least one more month.

## 6. Self-check

Five contract answers written before any query ran, not reverse engineered after seeing results. Grain proved, not assumed, row_count equal to distinct_key_count is a real check, not a claim. Availability checked with IS TRUE, not treated as probably fine. Every feature has an honest available when line, none of them reach past the 16th of March. The trap was performed, not just described, leak added, score jumped, leak removed, honest number kept. One limitation named plainly, not buried in a caveat.

Still open for later weeks: this label, within month first half versus second half, is a reasonable warehouse native replacement for the CSV's trend_direction, but it's still a short window. A prior ninety days to next thirty days version, the stretch goal I named in w02, is still the stronger target once I'm working across multiple months instead of one.